In [1]:
import imageio
import numpy as np
from tqdm import tqdm

import robosuite.macros as macros
from robosuite import make

# Set the image convention to opencv so that the images are automatically rendered "right side up" when using imageio
# (which uses opencv convention)
macros.IMAGE_CONVENTION = "opencv"

from robosuite.controllers import load_controller_config
from robosuite.utils.input_utils import *

In [2]:
options = {}
options["env_name"] = "PickPlace"
options["robots"] = "Panda"
# controller_name = "OSC_POSE"
# options["controller_configs"] = load_controller_config(default_controller=controller_name)

In [2]:
cameras = ['frontview', 'birdview', 'agentview', 'robot0_eye_in_hand']

In [4]:
# initialize the task
env = suite.make(
    **options,
    has_renderer=True,
    ignore_done=True,
    use_camera_obs=True,
    control_freq=20,
    camera_names=cameras,
)

# Get action limits
low, high = env.action_spec

In [5]:
for camera in cameras:
    writer = imageio.get_writer(camera + "_video.mp4", fps=20)

    frames = []
    obs = env.reset()
    for i in tqdm(range(10000)):
        action = np.random.uniform(low, high)
        obs, reward, done, info = env.step(action)
        
        # dump a frame from every K frames
        if i % 10 == 0:
            frame = obs[camera + "_image"]
            writer.append_data(frame)

        if done:
            break

    writer.close()

100%|██████████| 10000/10000 [04:12<00:00, 39.58it/s]


In [3]:
# view the trajectories!
from IPython.display import Video

def show_vid(name):
        return Video(name, embed=True)

(Video(camera + "_video.mp4", embed=True) for camera in cameras)

<generator object <genexpr> at 0x10b6a8e40>

In [4]:
from ipywidgets import Output, GridspecLayout
from IPython import display

grid = GridspecLayout(1, len(cameras))
print(cameras)
for i, camera in enumerate(cameras):
    out = Output()
    with out:
        display.display(display.Video(camera + "_video.mp4", embed=True))
    grid[0, i] = out

grid

['frontview', 'birdview', 'agentview', 'robot0_eye_in_hand']


GridspecLayout(children=(Output(layout=Layout(grid_area='widget001')), Output(layout=Layout(grid_area='widget0…

In [8]:
env.close_renderer()
env.close()